In [ ]:
!pip install transformers datasets torchmetrics

In [ ]:
import torch
import numpy as np

from datasets import load_dataset, Audio, DatasetDict

from transformers import AutoModel, AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Χρήση συσκευής: {device}")

In [ ]:
model_or = AutoModel.from_pretrained("facebook/hubert-base-ls960").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/hubert-finetuned-SAVEE").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/hubert-finetuned-SAVEE")

In [ ]:
dataset = load_dataset("AbstractTTS/SAVEE", split="train")
print(dataset[0])
print(dataset.features)

In [ ]:
def load_savee_speaker_independent():
    dataset = load_dataset("AbstractTTS/SAVEE", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        return example["file"].split("_")[0]

    train_speakers = ["DC", "JE"]
    test_speakers = ["JK", "KL"]

    train_dataset = dataset.filter(lambda x: get_speaker(x) in train_speakers)
    test_dataset = dataset.filter(lambda x: get_speaker(x) in test_speakers)

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_savee_speaker_independent()

In [ ]:
test_dataset = dataset["test"]

In [ ]:
def extract_embeddings(model, feature_extractor, dataset, num_samples=150, encoder_attr=None):
    probe_model = getattr(model, encoder_attr) if encoder_attr else model
    num_layers = model.config.num_hidden_layers + 1
    layer_embeddings = [[] for _ in range(num_layers)]

    for i in range(num_samples):
        sample = dataset[i]
        audio = sample["audio"]["array"]
        inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = probe_model(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(outputs.hidden_states):
            pooled = h.mean(dim=1)
            layer_embeddings[layer].append(pooled.squeeze().cpu().numpy())

    return [np.array(layer) for layer in layer_embeddings]

In [ ]:
emb_pretrained = extract_embeddings(model_or, feature_extractor, test_dataset, encoder_attr=None)
emb_finetuned = extract_embeddings(model_ft, feature_extractor, test_dataset, encoder_attr="hubert")

del model_or, model_ft
torch.cuda.empty_cache()

In [ ]:
#Συνάρτηση υπολογισμού του CKA
def linear_CKA(X, Y):

    X = X - X.mean(0)
    Y = Y - Y.mean(0)

    hsic = np.linalg.norm(X.T @ Y, ord="fro") ** 2

    var1 = np.linalg.norm(X.T @ X, ord="fro")
    var2 = np.linalg.norm(Y.T @ Y, ord="fro")

    return hsic / (var1 * var2)

In [ ]:
import pandas as pd

num_layers = len(emb_pretrained)
cka_scores = [linear_CKA(emb_pretrained[l], emb_finetuned[l]) for l in range(num_layers)]

cka_results = pd.DataFrame({"layer": range(num_layers), "cka": cka_scores})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cka_results["layer"], cka_results["cka"], marker="o", color="#065A82")
plt.xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
plt.ylabel("CKA similarity (Pretrained vs Fine-tuned)")
plt.title("Layer-wise CKA: HuBERT πριν/μετά το Fine-tuning (SAVEE)")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()